# Kircher ΔP: the CAV's own prediction of the mutation effect on the 21 saturation-mutagenesis regions

The Kircher counterpart of `Malinois MPRA 5`. For every SNV inside a motif occurrence of the 251 TFs with a function
CAV (NT-v2 500M, layer 24, the pooled HepG2 + K562 fit of `fcav_pool`, no refit), the occurrence's 600 bp window
(MPRA bases inside the region, hg38 outside, as in `fcav_pool/mpra_windows.json`) is embedded again with the
alternative base:

ΔP = P(bound | alt) − P(bound | ref), and **ΣΔP** = ΔP summed over all occurrences that contain the SNV (main score,
fixed in advance, as on Malinois). No PWM and no MPRA label is used.

Compared per region with motifbreakR (the call with the largest |alleleDiff|) and summed motifbreakR (alleleDiff
summed over calls), on the SNVs inside at least one occurrence (the set of `Pooled Regions 2`, whose motifbreakR
Pearson 0.177 / 0.279 / 0.464 for regions 0 / 8 / 20 must come back). Groups: the 4 regions measured in HepG2 or
K562, and the 17 others. Null N2: ΔP shuffled among the (SNV, occurrence) pairs of the same TF across all regions.
Also summed motifbreakR + ΣΔP, fitted leave-one-region-out. TF tiers by CAV AUC: all, ≥ 0.75.

**GPU runtime**, about 5–10 minutes in total (~1.4k occurrences, a few ×10k SNV–occurrence pairs).

In [ ]:
!pip install -q -U "transformers==4.57.6" "tokenizers==0.22.0" "huggingface_hub==0.34.0" "safetensors==0.4.5"
!pip install -q scikit-learn scipy pandas
from google.colab import drive
drive.mount("/content/drive")

## 1. Settings; readout copied unchanged from `Pooled Regions 1c`

In [ ]:
WORK_DIR = "/content/drive/MyDrive/GitHub/Biological-Foundation-Model/Notebooks/EP Pair Evaluation/per_occurrence/motif_token_study"
import os, json, time, gc, sys, re
import numpy as np, pandas as pd
import torch
from scipy import stats
from transformers import AutoTokenizer, AutoModel, AutoConfig, AutoModelForMaskedLM, AutoModelForCausalLM

C = json.load(open(f"{WORK_DIR}/config.json"))
POOL = f"{WORK_DIR}/fcav_pool"
OUT = f"{WORK_DIR}/kircher_deltaP"
os.makedirs(OUT, exist_ok=True)
MWJ = json.load(open(f"{POOL}/mpra_windows.json"))
INFO = {int(k): v for k, v in MWJ["regions"].items()}
TAG, LAYER = "nt500m", 24
REGISTRY = {"nt500m": dict(hf="InstaDeepAI/nucleotide-transformer-v2-500m-multi-species", tok_map=("kmer", 6),
                           causal=False, batch=32)}
FIT_LAYERS = {"nt500m": [LAYER]}
BASE_WEIGHTED = False
BATCH = 32
WLEN, MOFF = C["window_len"], C["motif_offset"]
FITD = f"{POOL}/fits/{TAG}"
DEV = torch.device("cuda")
COMP = str.maketrans("ACGTN", "TGCAN")
rc = lambda s: s.translate(COMP)[::-1]
HOME = {"HepG2", "K562"}
TIERS = [("all TFs", 0.0), ("AUC >= 0.75", 0.75)]
N_PERM = 1000
MBR_THRESH, MBR_NEUT = 0.85, 0.4
pd.set_option("display.width", 250)
fmt = lambda x: f"{x:+.3f}"

In [ ]:
def load_tokenizer(tag):
    spec = REGISTRY[tag]
    tokz = AutoTokenizer.from_pretrained(spec["hf"], trust_remote_code=True)
    tokz.padding_side = "right"          # several of these default to left (HyenaDNA)
    if tokz.pad_token is None:
        tokz.pad_token = tokz.eos_token if tokz.eos_token is not None else tokz.unk_token
    return tokz


def token_bases(spec, n):
    # (n, 2) base interval of each sequence token, for the arithmetic maps
    kind = spec["tok_map"][0]
    if kind == "kmer":
        k = spec["tok_map"][1]
        return np.stack([np.arange(n) * k, (np.arange(n) + 1) * k], 1)
    if kind == "char":
        return np.stack([np.arange(n), np.arange(n) + 1], 1)
    raise RuntimeError("offsets are resolved by the tokenizer, not here")


def encode(tokz, spec, seqs):
    # input_ids (B, T), attention mask (B, T), and per sequence: the positions of its sequence tokens in input_ids
    # and the base interval each one covers
    kind = spec["tok_map"][0]
    if kind == "offsets":
        assert tokz.is_fast, f"{spec['hf']} has no fast tokenizer, so offset mapping is unavailable"
        enc = tokz(seqs, return_tensors="pt", padding=True, return_offsets_mapping=True)
    else:
        enc = tokz(seqs, return_tensors="pt", padding=True)
    ids = enc["input_ids"]
    am = enc.get("attention_mask")
    if am is None:
        am = (ids != tokz.pad_token_id).long()
    amn = am.numpy()
    pos, spans = [], []
    if kind == "offsets":
        om = enc["offset_mapping"].numpy()
        for b in range(len(seqs)):
            keep = (om[b, :, 1] > om[b, :, 0]) & (amn[b] == 1)
            pos.append(np.flatnonzero(keep))
            spans.append(om[b, keep])
    else:
        for b in range(len(seqs)):
            sp = np.array(tokz.get_special_tokens_mask(ids[b].tolist(), already_has_special_tokens=True))
            p = np.flatnonzero((sp == 0) & (amn[b] == 1))
            pos.append(p)
            spans.append(token_bases(spec, len(p)))
        assert len({len(p) for p in pos}) == 1, f"{spec['hf']}: windows of equal length gave different token counts"
    return ids, am, pos, spans


def span_weights(bases, lo, hi):
    # per-token weight for the base interval [lo, hi)
    ov = np.minimum(bases[:, 1], hi) - np.maximum(bases[:, 0], lo)
    ov = np.clip(ov, 0, None).astype(np.float32)
    if not BASE_WEIGHTED:
        ov = (ov > 0).astype(np.float32)
    assert ov.sum() > 0, f"no token covers bases [{lo}, {hi})"
    return ov / ov.sum()


def load_model(tag):
    spec = REGISTRY[tag]
    # the class the checkpoint ships its own code for (NT-v2 has AutoModelForMaskedLM but no AutoModel)
    amap = getattr(AutoConfig.from_pretrained(spec["hf"], trust_remote_code=True), "auto_map", None) or {}
    cls = next((c for k, c in (("AutoModel", AutoModel), ("AutoModelForMaskedLM", AutoModelForMaskedLM),
                               ("AutoModelForCausalLM", AutoModelForCausalLM)) if k in amap), AutoModel)
    mdl = cls.from_pretrained(spec["hf"], trust_remote_code=True).to(DEV).eval()
    # DNABERT-2's triton flash-attention kernel does not compile on current triton; its code falls back to PyTorch
    for name in {type(m).__module__ for m in mdl.modules()}:
        mod = sys.modules.get(name)
        if mod is not None and getattr(mod, "flash_attn_qkvpacked_func", None) is not None:
            mod.flash_attn_qkvpacked_func = None
            print(f"{tag}: triton flash-attention disabled in {name.split('.')[-1]}; using PyTorch attention")
    if spec.get("out") == "dnabert2" and getattr(mdl, "pooler", None) is not None:
        mdl.pooler = None
    return mdl


POS, DEPTH, LOADED = {}, {}, {}


@torch.no_grad()
def hidden_stack(tag, model, ids, am):
    # one (layers, T_j, H) float tensor per sequence; only the layers in POS[tag] once prepare() has set it
    spec = REGISTRY[tag]
    if spec.get("out") == "dnabert2":
        # DNABERT-2 returns every encoder layer unpadded; one sequence at a time keeps the token order
        out = []
        for j in range(ids.shape[0]):
            t = ids[j][am[j].bool()][None].to(DEV)
            layers, _ = model(t, attention_mask=torch.ones_like(t), output_all_encoded_layers=True)
            S = torch.stack([x.reshape(t.shape[1], -1) for x in layers], 0).float()
            out.append(S[POS[tag]] if tag in POS else S)
        return out
    kw = {} if spec["causal"] else {"attention_mask": am.to(DEV)}
    hs = model(input_ids=ids.to(DEV), output_hidden_states=True, **kw).hidden_states
    if tag in POS:
        hs = [hs[i] for i in POS[tag]]
    S = torch.stack(hs, 1).float()
    return [S[j] for j in range(S.shape[0])]


PROBE = [h["window"] for h in MWJ["hits"][:4]]


def prepare(tag):
    # load the model; from one forward pass: number of layers, token tiling, and where FIT_LAYERS sit in the stack
    spec = REGISTRY[tag]
    mdl = load_model(tag)
    tokz = LOADED.get(tag, {}).get("tokz") or load_tokenizer(tag)
    LOADED[tag] = dict(spec=spec, tokz=tokz)
    POS.pop(tag, None)
    ids, am, pos, spans = encode(tokz, spec, PROBE[:2])
    S = hidden_stack(tag, mdl, ids, am)
    for j, Sj in enumerate(S):
        assert Sj.ndim == 3 and int(pos[j].max()) < Sj.shape[1]
        b = spans[j]
        assert b[0, 0] == 0 and b[-1, 1] == WLEN and (b[1:, 0] == b[:-1, 1]).all(), f"{tag}: tokens do not tile"
    nl = S[0].shape[0]
    lids = list(range(1, nl + 1)) if spec.get("out") == "dnabert2" else list(range(nl))   # DNABERT-2: no embedding layer
    DEPTH[tag] = lids[-1]
    miss = [l for l in FIT_LAYERS[tag] if l not in lids]
    assert not miss, f"{tag}: layers {miss} do not exist (ids {lids[0]}..{lids[-1]})"
    POS[tag] = [lids.index(l) for l in FIT_LAYERS[tag]]
    print(f"{tag}: {nl} layers (depth {DEPTH[tag]}), fitting {FIT_LAYERS[tag]}, hidden {S[0].shape[2]}, "
          f"{len(pos[0])} tokens for {WLEN} bp")
    del S
    return mdl


@torch.no_grad()
def readout(tag, model, seqs, lohi):
    # (n, layers, D): D = hidden, or 2 * hidden for a causal model (forward ‖ reverse complement)
    L = LOADED[tag]
    tokz, spec = L["tokz"], L["spec"]
    passes = [(seqs, lohi)]
    if spec["causal"]:
        passes.append(([rc(s) for s in seqs], [(WLEN - b, WLEN - a) for a, b in lohi]))
    bs = spec.get("batch", BATCH)
    parts = []
    for ss, lh in passes:
        out = None
        for i in range(0, len(ss), bs):
            chunk, lhc = ss[i:i + bs], lh[i:i + bs]
            ids, am, pos, spans = encode(tokz, spec, chunk)
            S = hidden_stack(tag, model, ids, am)
            if out is None:
                out = np.zeros((len(ss), S[0].shape[0], S[0].shape[2]), np.float16)
            for j in range(len(chunk)):
                w = torch.tensor(span_weights(spans[j], *lhc[j]), device=DEV)
                p = torch.tensor(pos[j], device=DEV)
                out[i + j] = (S[j][:, p] * w[None, :, None]).sum(1).half().cpu().numpy()
            del S
        parts.append(out)
    if len(parts) == 1:
        return parts[0]
    if spec.get("rc_mode", "concat") == "mean":
        return ((parts[0].astype(np.float32) + parts[1].astype(np.float32)) / 2).astype(np.float16)
    return np.concatenate(parts, 2)

## 2. P(bound | ref) of every occurrence and P(bound | alt) of every (SNV, occurrence) pair

In [ ]:
def direction(m):
    z = np.load(f"{FITD}/{m}.npz", allow_pickle=False)
    li = [int(l) for l in z["layers"]].index(LAYER)
    return z, li, z["w_all"][li].astype(np.float64), z["mu"][li].astype(np.float64), z["sd"][li].astype(np.float64)


def decision(m, windows, w_motif, mdl):
    _, _, w, mu, sd = direction(m)
    X = readout(TAG, mdl, windows, [(MOFF, MOFF + w_motif)] * len(windows))[:, 0].astype(np.float64)
    return ((X - mu) / sd) @ w[:-1] + w[-1]


HAVE = {f[:-4] for f in os.listdir(FITD) if f.endswith(".npz") and not f.endswith(".part.npz")}
HW = [h for h in MWJ["hits"] if h["motif"] in HAVE]
WIN = {}
for h in MWJ["hits"]:
    WIN.setdefault(h["region"], []).append(h)
rows = []
for r in sorted({h["region"] for h in HW}):
    S_ = f"{POOL}/regions/r{r}"
    snv = pd.read_csv(f"{S_}/snvs.csv")
    q = pd.read_csv(f"{S_}/snv_hit_dpwm.csv")
    hr = pd.DataFrame([{k: h[k] for k in ("hit_id", "motif", "start", "end")} for h in HW if h["region"] == r])
    q = q.merge(hr, on="hit_id")
    q["region"] = r
    q["local"], q["ref"], q["alt"] = snv.loc[q["snv"], "local"].values, snv.loc[q["snv"], "ref"].values, snv.loc[q["snv"], "alt"].values
    rows.append(q)
PAIRS = pd.concat(rows, ignore_index=True)
HIT = {(h["region"], h["hit_id"]): h for h in HW}
print(f"{len(HW):,} occurrences of {len({h['motif'] for h in HW})} TFs in {len({h['region'] for h in HW})} regions; "
      f"{len(PAIRS):,} (SNV, occurrence) pairs")


def alt_window(row):
    h = HIT[row.region, row.hit_id]
    k = int(row.local) - (h["start"] - MOFF)
    x = h["window"]
    assert 0 <= k < WLEN and x[k] == row.ref, f"region {row.region} hit {row.hit_id}: reference base not at {k}"
    return x[:k] + row.alt + x[k + 1:]


mdl = prepare(TAG)
t0 = time.time()
PREF, chk = {}, []
for m in sorted({h["motif"] for h in HW}):
    hs = [h for h in HW if h["motif"] == m]
    d = decision(m, [h["window"] for h in hs], hs[0]["end"] - hs[0]["start"], mdl)
    z, li, *_ = direction(m)
    oof = pd.Series(z["prob_hit"][:, li], index=pd.MultiIndex.from_arrays([z["hit_region"], z["hit_id"]]))
    for h, v in zip(hs, d):
        PREF[h["region"], h["hit_id"]] = 1 / (1 + np.exp(-v))
        chk.append((PREF[h["region"], h["hit_id"]], oof.get((h["region"], h["hit_id"]), np.nan)))
chk = np.array(chk)
ok = np.isfinite(chk[:, 1])
print(f"check: P(bound | ref) with the all-sites direction vs the stored out-of-fold P, {ok.sum()} occurrences: "
      f"Spearman {stats.spearmanr(chk[ok, 0], chk[ok, 1])[0]:.3f}")
DEC = np.zeros(len(PAIRS))
for m, g in PAIRS.groupby("motif"):
    DEC[g.index] = decision(m, [alt_window(r_) for r_ in g.itertuples()], int(g["end"].iloc[0] - g["start"].iloc[0]), mdl)
PAIRS["P_ref"] = [PREF[r, h] for r, h in zip(PAIRS["region"], PAIRS["hit_id"])]
PAIRS["P_alt"] = 1 / (1 + np.exp(-DEC))
PAIRS["dP"] = PAIRS["P_alt"] - PAIRS["P_ref"]
PAIRS.drop(columns=["start", "end"]).to_csv(f"{OUT}/pairs.csv", index=False)
print(f"done [{(time.time() - t0) / 60:.1f} min]; ΔP: mean {PAIRS['dP'].mean():+.4f}, sd {PAIRS['dP'].std():.4f}; saved {OUT}/pairs.csv")
del mdl; gc.collect(); torch.cuda.empty_cache()

## 3. Per region: motifbreakR, summed motifbreakR, ΣΔP (motifbreakR code copied from the diagnosis notebook)

In [ ]:
def _read_meme(path):
    out, name, rows_, ns = {}, None, [], 20
    for line in open(path):
        t = line.split()
        if line.startswith("MOTIF"):
            if name is not None and name not in out:
                out[name] = (np.array(rows_), ns)
            name, rows_, ns = (t[2] if len(t) > 2 else t[1]), [], 20
        elif line.startswith("letter-probability"):
            mm = re.search(r"nsites=\s*([0-9.]+)", line)
            ns = float(mm.group(1)) if mm else 20
        elif name is not None and len(t) == 4 and all(re.fullmatch(r"[0-9.eE+-]+", x) for x in t):
            rows_.append([float(x) for x in t])
    if name is not None and name not in out:
        out[name] = (np.array(rows_), ns)
    return out


MBR_MATS = {}
for name, (f, ns) in _read_meme(C["meme_path"]).items():
    f = f / f.sum(1, keepdims=True)
    pp = (f * ns + 0.25) / (ns + 1)
    S = pp * (pp.max(1) - pp.min(1))[:, None]
    S = np.vstack([S.T, np.zeros(len(S))]).T
    MBR_MATS[name] = (S, S[::-1][:, [3, 2, 1, 0, 4]], S[:, :4].min(1).sum(), S[:, :4].max(1).sum())
ENC = {"A": 0, "C": 1, "G": 2, "T": 3}
WIN = {}
for h in MWJ["hits"]:
    WIN.setdefault(h["region"], []).append(h)


def build(r, HS):
    # verbatim from Pooled Regions 2: SNV x hit rows and motifbreakR calls on the hits HS of region r
    S_ = f"{POOL}/regions/r{r}"
    hits = pd.read_csv(f"{S_}/hits.csv").set_index("hit_id")
    snv = pd.read_csv(f"{S_}/snvs.csv")
    q = pd.read_csv(f"{S_}/snv_hit_dpwm.csv")
    q = q[q["hit_id"].isin(HS)].sort_values(["snv", "hit_id"]).reset_index(drop=True)
    sn = q["snv"].values
    st = np.r_[0, np.flatnonzero(sn[1:] != sn[:-1]) + 1]
    hi = np.searchsorted(HS, q["hit_id"].values)
    n_snv = len(st)
    MOT = hits.loc[HS, "motif"].values
    base = {}
    for h in WIN[r]:
        for i, b in enumerate(h["window"]):
            base.setdefault(h["start"] - C["motif_offset"] + i, b)
    lo = min(base)
    seq = np.array([ENC.get(base.get(x, "N"), 4) for x in range(lo, max(base) + 1)])
    s_ = snv.loc[sn[st]]
    pos, ref, alt = s_["local"].values - lo, s_["ref"].map(ENC).values, s_["alt"].map(ENC).values
    assert (seq[pos] == ref).all(), "rebuilt sequence disagrees with the SNV reference bases"
    out = []
    for m in sorted(set(MOT)):
        if m not in MBR_MATS:
            continue
        S, Sr, smin, smax = MBR_MATS[m]
        w = len(S)
        n = len(seq) - w + 1
        idx = np.arange(n)[:, None] + np.arange(w)[None, :]
        f_fwd, f_rev = S[np.arange(w), seq[idx]].sum(1), Sr[np.arange(w), seq[idx]].sum(1)
        stw = pos[:, None] - np.arange(w)[None, :]
        k = np.broadcast_to(np.arange(w), stw.shape)
        ok = (stw >= 0) & (stw < n)
        stc = np.clip(stw, 0, n - 1)
        ref_f, ref_r = np.where(ok, f_fwd[stc], -np.inf), np.where(ok, f_rev[stc], -np.inf)
        alt_f = ref_f + np.where(ok, S[k, alt[:, None]] - S[k, ref[:, None]], 0)
        alt_r = ref_r + np.where(ok, Sr[k, alt[:, None]] - Sr[k, ref[:, None]], 0)
        sr, sa = np.maximum(ref_f, ref_r).max(1), np.maximum(alt_f, alt_r).max(1)
        pr, pa = (sr - smin) / (smax - smin), (sa - smin) / (smax - smin)
        diff = sa - sr
        keep = np.isfinite(sr) & ((pr > MBR_THRESH) | (pa > MBR_THRESH)) & (np.abs(diff) >= MBR_NEUT)
        out += [(i, m, diff[i]) for i in np.flatnonzero(keep)]
    t = pd.DataFrame(out, columns=["i", "motif", "alleleDiff"])
    t["j"] = np.arange(len(t))
    rows_ = pd.DataFrame({"i": np.repeat(np.arange(n_snv), np.diff(np.r_[st, len(sn)])), "motif": MOT[hi], "h": hi})
    link = t[["j", "i", "motif"]].merge(rows_, on=["i", "motif"])
    keep = np.sort(link["j"].unique())
    t = t.loc[keep].reset_index(drop=True)
    return dict(snv=snv, n_snv=n_snv, snv_id=sn[st], y=snv.loc[sn[st], "log2FC"].values,
                DIFF=t["alleleDiff"].values, TI=t["i"].values, MOTP=t["motif"].values)


z_auc = {}
for m in HAVE:
    z = np.load(f"{FITD}/{m}.npz", allow_pickle=False)
    z_auc[m] = float(np.asarray(z["auc"])[[int(l) for l in z["layers"]].index(LAYER)])
PAIRS["auc"] = PAIRS["motif"].map(z_auc)


def pear(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    if a.std() == 0 or b.std() == 0:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def region_data(r, thr):
    hs = np.array(sorted(h["hit_id"] for h in HW if h["region"] == r and z_auc[h["motif"]] >= thr))
    if not len(hs):
        return None
    R = build(r, hs)
    n = R["n_snv"]
    o = np.lexsort((-np.abs(R["DIFF"]), R["TI"]))
    fb = o[np.r_[True, R["TI"][o][1:] != R["TI"][o][:-1]]] if len(o) else o
    ad = np.zeros(n); ad[R["TI"][fb]] = R["DIFF"][fb]
    sad = np.bincount(R["TI"], R["DIFF"], minlength=n)
    pos = pd.Series(np.arange(n), index=R["snv_id"])
    pr = PAIRS[(PAIRS["region"] == r) & PAIRS["hit_id"].isin(hs)]
    return dict(r=r, home=set(INFO[r]["cell"].split(",")) <= HOME, y=R["y"], ad=ad, sad=sad,
                pidx=pos.reindex(pr["snv"]).values.astype(int), pair_rows=pr.index.values)


def sdp(D, dP):
    return np.bincount(D["pidx"], dP[D["pair_rows"]], minlength=len(D["y"]))


RES, NULLS = {}, {}
for tier, thr in TIERS:
    DS = [d for d in (region_data(r, thr) for r in sorted(INFO)) if d is not None and len(d["y"]) >= 20]
    dP = PAIRS["dP"].values
    rows = []
    for d in DS:
        s = sdp(d, dP)
        rows.append({"region": d["r"], "cell": INFO[d["r"]]["cell"], "home": d["home"], "SNVs in hits": len(d["y"]),
                     "motifbreakR": pear(d["ad"], d["y"]), "summed motifbreakR": pear(d["sad"], d["y"]), "ΣΔP": pear(s, d["y"])})
    T = pd.DataFrame(rows).set_index("region")
    RES[tier] = (T, DS)
    if tier == "all TFs":
        print("check against Pooled Regions 2 (motifbreakR): " + ", ".join(
            f"region {r} {T.loc[r, 'motifbreakR']:.3f} (was {w})" for r, w in ((0, 0.177), (8, 0.279), (20, 0.464)) if r in T.index))
    print(f"\n=== {tier}: Pearson with the measured log2FC, per region\n")
    print(T.to_string(float_format=fmt))

## 4. Summary over regions, the shuffle null, and summed motifbreakR + ΣΔP

Mean Pearson over the regions of each group. p N2: share of 1000 shuffles of ΔP among the pairs of the same TF
(across all regions) whose mean ΣΔP Pearson is at least as large. "summed motifbreakR + ΣΔP": log2FC standardised
within region, a linear fit on all other regions of the group applied to the left-out region (leave-one-region-out),
against the same with summed motifbreakR alone.

In [ ]:
cm = pd.factorize(PAIRS["motif"])[0]
base = np.argsort(cm, kind="stable")


def shuffle_tf(rng):
    dP = PAIRS["dP"].values
    o = np.lexsort((rng.random(len(dP)), cm))
    out = np.empty_like(dP); out[base] = dP[o]
    return out


def loro(DS, cols):
    zs = lambda v: (v - v.mean()) / v.std()
    out = []
    for i, d in enumerate(DS):
        tr = [e for j, e in enumerate(DS) if j != i]
        Ztr = np.column_stack([np.ones(sum(len(e["y"]) for e in tr))] + [np.concatenate([c(e) for e in tr]) for c in cols])
        b = np.linalg.lstsq(Ztr, np.concatenate([zs(e["y"]) for e in tr]), rcond=None)[0]
        out.append(pear(np.column_stack([np.ones(len(d["y"]))] + [c(d) for c in cols]) @ b, d["y"]))
    return np.array(out)


rows = []
rng = np.random.default_rng(7)
for tier, (T, DS) in RES.items():
    null = []
    for _ in range(N_PERM):
        dPs = shuffle_tf(rng)
        null.append({h: np.nanmean([pear(sdp(d, dPs), d["y"]) for d in DS if d["home"] == h]) for h in (True, False)})
    dP = PAIRS["dP"].values
    for h in (True, False):
        G = [d for d in DS if d["home"] == h]
        if len(G) < 3:
            continue
        t = T[T["home"] == h]
        obs = t["ΣΔP"].mean()
        a0 = loro(G, [lambda e: e["sad"]])
        a1 = loro(G, [lambda e: e["sad"], lambda e: sdp(e, dP)])
        rows.append({"tier": tier, "regions": f"{'HepG2/K562' if h else 'other cells'} ({len(G)})",
                     "motifbreakR": t["motifbreakR"].mean(), "summed motifbreakR": t["summed motifbreakR"].mean(),
                     "ΣΔP": obs, "p N2 ΣΔP": np.mean([n[h] >= obs for n in null]),
                     "ΣΔP beats summed in": f"{int((t['ΣΔP'] > t['summed motifbreakR']).sum())}/{len(t)}",
                     "summed (LORO)": np.nanmean(a0), "summed + ΣΔP (LORO)": np.nanmean(a1),
                     "+ΣΔP wins in": f"{int((a1 > a0).sum())}/{len(G)}"})
S = pd.DataFrame(rows).set_index(["tier", "regions"])
S.to_csv(f"{OUT}/summary.csv")
for tier, (T, _) in RES.items():
    T.to_csv(f"{OUT}/per_region_{tier.replace(' ', '_').replace('>=', 'ge')}.csv")
print("mean Pearson over regions\n")
print(S.to_string(float_format=fmt))

In [ ]:
from google.colab import runtime
runtime.unassign()